In [1]:
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)

from common.const import (
    START_TIMESTAMP_5CORE_TRAIN, END_TIMESTAMP_5CORE_TRAIN,
    SCHEMA, TABLE_RAW_PRODUCTS, TABLE_RAW_REVIEWS, TABLE_5CORE_TIMESTAMP_W_HIS_DICT, TABLE_5CORE_TIMESTAMP_W_HIS_TRAIN_USER_ITEM_INTERACTIONS, TABLE_RECENT_ITEM_REVIEW_SUMMARY
)
from common.helper import get_categories, connect_clickhouse, load_hf_data, parse_price, parse_product_images, parse_review_images, measure_clickhouse_query_performance, clickhouse_extract_data_as_list, convert_list_of_values_to_string, clickhouse_extract_data_as_df
import json
from tqdm import tqdm
import numpy as np
import pandas as pd

categories = get_categories()
categories = categories[:10] # This line is for testing only (selecting a subset of categories).
BATCH_SIZE = 10000

E0000 00:00:1772818613.225385 1213022 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1772818613.225412 1213022 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_rejected' registered more than once. Ignoring later registration.
E0000 00:00:1772818613.225414 1213022 instrument.cc:563] Metric with name 'grpc.resource_quota.connections_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1772818613.225415 1213022 instrument.cc:563] Metric with name 'grpc.resource_quota.instantaneous_memory_pressure' registered more than once. Ignoring later registration.
E0000 00:00:1772818613.225416 1213022 instrument.cc:563] Metric with name 'grpc.resource_quota.memory_pressure_control_value' registered more than once. Ignoring later registration.


# Create schema and tables

Since ClickHouse does not allow multi-statements, the SQL code needs to be separated.

In [4]:
queries = [
    f"CREATE DATABASE IF NOT EXISTS {SCHEMA};", # Create database
    f"DROP TABLE IF EXISTS {SCHEMA}.{TABLE_RAW_REVIEWS};", # Drop table
    
    
    f"""CREATE TABLE IF NOT EXISTS {SCHEMA}.{TABLE_RAW_REVIEWS}
    (
        timestamp UInt64,
        user_id String,
        parent_asin String,
        asin String,
        rating UInt8,
        title String,
        text String,
        first_image Nullable(String),
        image_count Int32,
        helpful_vote Int32,
        verified_purchase Bool,
        product_category String
    )
    ENGINE = MergeTree()
    ORDER BY (parent_asin, timestamp)
    PRIMARY KEY (parent_asin, timestamp);""", # Create reviews table
    
    
    f"DROP TABLE IF EXISTS {SCHEMA}.{TABLE_RAW_PRODUCTS};", # Drop table
    f"""CREATE TABLE IF NOT EXISTS {SCHEMA}.{TABLE_RAW_PRODUCTS}
    (
        main_category Nullable(String),
        title String,
        average_rating Float32,
        rating_number Int32,
        features Nullable(String),
        description Nullable(String),
        price Nullable(Decimal(10, 2)),
        first_image Nullable(String),
        image_count Int32,
        variant_count Int32,
        videos String,
        store Nullable(String),
        categories Nullable(String),
        details Nullable(String),
        parent_asin String,
        bought_together Nullable(String),
        subtitle Nullable(String),
        author Nullable(String),
        product_category String
    )
    ENGINE = MergeTree()
    ORDER BY parent_asin
    PRIMARY KEY parent_asin;""" # Create products table
]

client = connect_clickhouse()

for i, query in enumerate(tqdm(queries, desc="Setting up ClickHouse")):
    client.command(query)
    print(f"Query {i+1}: Successful")
    
client.close()

Setting up ClickHouse:  60%|██████    | 3/5 [00:00<00:00, 19.75it/s]

Query 1: Successful
Query 2: Successful
Query 3: Successful
Query 4: Successful


Setting up ClickHouse: 100%|██████████| 5/5 [00:00<00:00, 17.44it/s]

Query 5: Successful


# Import data

## Import data to `raw_products` table

In [5]:
client = connect_clickhouse()

columns = [
    "main_category",
    "title",
    "average_rating",
    "rating_number",
    "features",
    "description",
    "price",
    "first_image",
    "image_count",
    "variant_count",
    "videos",
    "store",
    "categories",
    "details",
    "parent_asin",
    "bought_together",
    "subtitle",
    "author",
    "product_category"
]

batch = []

for category in categories:
    data = load_hf_data(
        root_split="raw",
        data_type="meta",
        category=category,
    )
    for record in tqdm(data, desc=f"Loading data from {category} category..."):
        title = record['title']
        if title is not None:
            title = title.strip()
        
        first_image, image_count, variant_count = parse_product_images(record['images'])
        
        details = json.loads(record["details"])
        if len(details) == 0:
            details = None
        else:
            details = json.dumps(details).strip()
        
        batch.append([
            record["main_category"],
            title,
            record["average_rating"],
            record["rating_number"],
            convert_list_of_values_to_string(record["features"]),
            convert_list_of_values_to_string(record["description"]),
            parse_price(record["price"]),
            first_image,
            image_count,
            variant_count,
            json.dumps(record["videos"]),
            record["store"],
            convert_list_of_values_to_string(record["categories"]),
            details,
            record["parent_asin"],
            record["bought_together"],
            record["subtitle"],
            record["author"],
            category
        ])

        if len(batch) >= BATCH_SIZE:
            client.insert(
                f"{SCHEMA}.{TABLE_RAW_PRODUCTS}",
                batch,
                column_names=columns
            )
            batch.clear()

    # Insert remaining rows
    if batch:
        client.insert(
            f"{SCHEMA}.{TABLE_RAW_PRODUCTS}",
            batch,
            column_names=columns
        )
        batch.clear()
        
client.close()

Loading data from All_Beauty category...: 112590it [00:56, 1980.69it/s]
Loading data from Toys_and_Games category...: 890874it [09:30, 1562.15it/s]
Loading data from Cell_Phones_and_Accessories category...: 1288490it [13:15, 1620.52it/s]
Loading data from Industrial_and_Scientific category...: 427564it [04:07, 1730.38it/s]
Loading data from Gift_Cards category...: 1137it [00:00, 1748.44it/s]
Loading data from Musical_Instruments category...: 213593it [02:04, 1709.64it/s]
Loading data from Electronics category...: 1610012it [16:33, 1619.89it/s]
Loading data from Handmade_Products category...: 164817it [01:38, 1681.11it/s]
Loading data from Arts_Crafts_and_Sewing category...: 801446it [07:36, 1754.26it/s]
Loading data from Baby_Products category...: 217724it [02:14, 1624.16it/s]


## Import data to `raw_reviews` table

In [6]:
client = connect_clickhouse()

columns = [
    "timestamp",
    "user_id",
    "parent_asin",
    "asin",
    "rating",
    "title",
    "text",
    "first_image",
    "image_count",
    "helpful_vote",
    "verified_purchase",
    "product_category"
]

batch = []

for category in categories:
    data = load_hf_data(
        root_split="raw",
        data_type="review",
        category=category,
    )

    for record in tqdm(data, desc=f"Loading data from {category} category..."):
        title = record['title']
        if title is not None:
            title = title.strip()
        
        text = record['text']
        if text is not None:
            text = text.strip()
        
        first_image, image_count = parse_review_images(record["images"])
        
        batch.append([
            record["timestamp"],
            record["user_id"],
            record["parent_asin"],
            record["asin"],
            record["rating"],
            title,
            text,
            first_image,
            image_count,
            record["helpful_vote"],
            record["verified_purchase"],
            category
        ])

        if len(batch) >= BATCH_SIZE:
            client.insert(
                f"{SCHEMA}.{TABLE_RAW_REVIEWS}",
                batch,
                column_names=columns
            )
            batch.clear()

    # Insert remaining rows
    if batch:
        client.insert(
            f"{SCHEMA}.{TABLE_RAW_REVIEWS}",
            batch,
            column_names=columns
        )
        batch.clear()
        
client.close()

Loading data from All_Beauty category...: 701528it [00:40, 17117.39it/s]
Loading data from Toys_and_Games category...: 16260406it [14:59, 18069.97it/s]
Loading data from Cell_Phones_and_Accessories category...: 20812945it [19:19, 17950.29it/s]
Loading data from Industrial_and_Scientific category...: 5183005it [07:19, 11787.61it/s]
Loading data from Gift_Cards category...: 152410it [00:08, 17177.17it/s]
Loading data from Musical_Instruments category...: 3017439it [04:09, 12103.48it/s]
Loading data from Electronics category...: 43886944it [59:39, 12261.75it/s]
Loading data from Handmade_Products category...: 664162it [00:47, 13921.12it/s]
Loading data from Arts_Crafts_and_Sewing category...: 8966758it [10:49, 13809.57it/s]
Loading data from Baby_Products category...: 6028884it [08:04, 12445.25it/s]


# Measure query performance

## Set up

In [5]:
num_iter = 10

# Randomly select products
# query = f"""
# SELECT parent_asin
# FROM {SCHEMA}.{TABLE_RAW_PRODUCTS}
# ORDER BY rand()
# LIMIT {num_iter}
# """
# random_product_list = clickhouse_extract_data_as_list(query)
random_product_list = ['B07484MS4G', 'B01FYDF3VI', 'B01C3PMM7S', 'B012HUTKTM', 'B075R6NQ1W', 'B08DCZGQJ5', 'B077WBYPLG', 'B001KYU8UM', 'B0919HCJDN', 'B08MQCXGJM']


# Random keywords
# query = f"""
# SELECT title, description
# FROM {SCHEMA}.{TABLE_RAW_PRODUCTS}
# WHERE
#     description IS NOT NULL
#     AND title IS NOT NULL
#     AND length(splitByWhitespace(assumeNotNull(description))) > 10
#     AND length(splitByWhitespace(assumeNotNull(title))) > 5
# ORDER BY rand()
# LIMIT {num_iter}
# """
# query_output = clickhouse_extract_data_as_df(query)
# with pd.option_context('display.max_colwidth', None):
#     display(query_output['description'])

random_keywords = [
    "Plumping Whip", "Eye Sticker", # 2
    "Hair Scrunchies Colorful", "facial jade roller", # 3
    "Professional Supreme Performance Braiding", "vintage English floral fragrance", "Water Weightless Hydration Conditioner", # 4
    "hottest beauty product of 2019", "Mist Coolant Lubrication Spray System", "keep the skin calm and soothed" # 5
]

# Random users
# query = f"""
# SELECT user_id
# FROM {SCHEMA}.{TABLE_RAW_REVIEWS}
# ORDER BY rand()
# LIMIT {num_iter}
# """
# random_user_list = clickhouse_extract_data_as_list(query)
random_user_list = ['AG5JJZKUDTC5PREHPTI5NOC75M2Q', 'AFQIHDRZZ7RGRLSFKDGCYPBHHTKA', 'AFXCPSCWLOSVQZHN7FAECOYAT3RA', 'AHT6VS3G3JSA7KBOCBT6KPUCJX7A', 'AH4G4SNKSIWZNGZ5EDNJTRORKIMQ', 'AG4JLUSDCYTYSSE5SQADXOPATLTA', 'AGXVAAK6XJ7C2GUTYQEMBOGQ7JGQ', 'AGKPEXI6SAACYVXRNMICW3NKG2XA', 'AHA272AOVKWZUXL672G653A2MTIA', 'AH6X5VUV5AUGO3G55G7AL6GZ2QZQ']

## Query 1: Given a product ID (parent_asin), retrieve the product title and full product description.

In [6]:
# Record query duration
query_duration_logs = []
for item in random_product_list:
    query = f"""
    SELECT
        title,
        description
    FROM {SCHEMA}.{TABLE_RAW_PRODUCTS}
    WHERE parent_asin = '{item}';
    """

    query_duration, _ = measure_clickhouse_query_performance(query)
    query_duration_logs.append(query_duration)
    
print(f"Over {len(random_product_list)} runs:")
print(f"Median query duration: {np.median(query_duration_logs)} ms")
print(f"Mean query duration: {np.mean(query_duration_logs)} ms")

Over 10 runs:
Median query duration: 8.0 ms
Mean query duration: 8.5 ms


## Query 2: Given a product ID (parent_asin), retrieve the most recent N customer reviews, including rating, review title, review text, and timestamp.

In [7]:
# Record query duration
query_duration_logs = []
for item in random_product_list:
    query = f"""
    SELECT
        timestamp,
        title,
        text,
        rating
    FROM {SCHEMA}.{TABLE_RAW_REVIEWS}
    WHERE parent_asin = '{item}'
    ORDER BY timestamp DESC
    LIMIT 10;
    """

    query_duration, _ = measure_clickhouse_query_performance(query)
    query_duration_logs.append(query_duration)
    
print(f"Over {len(random_product_list)} runs:")
print(f"Median query duration: {np.median(query_duration_logs)} ms")
print(f"Mean query duration: {np.mean(query_duration_logs)} ms")

Over 10 runs:
Median query duration: 11.5 ms
Mean query duration: 12.1 ms


## Query 3: Given a keyword, return all products whose title or description contains the keyword.

In [8]:
# Record query duration
query_duration_logs = []
for item in random_keywords:
    query = f"""
    SELECT
        parent_asin,
        title,
        description
    FROM {SCHEMA}.{TABLE_RAW_PRODUCTS}
    WHERE
        (positionCaseInsensitive(assumeNotNull(title), '{item}') > 0)
        OR
        (positionCaseInsensitive(assumeNotNull(description), '{item}') > 0)
    """

    query_duration, _ = measure_clickhouse_query_performance(query)
    query_duration_logs.append(query_duration)
    
print(f"Over {len(random_product_list)} runs:")
print(f"Median query duration: {np.median(query_duration_logs)} ms")
print(f"Mean query duration: {np.mean(query_duration_logs)} ms")

Over 10 runs:
Median query duration: 66.5 ms
Mean query duration: 67.7 ms


## Query 4: Given a user ID, retrieve all products reviewed by that user along with the corresponding ratings and review dates.

In [9]:
# Record query duration
query_duration_logs = []
for item in random_user_list:
    query = f"""
    SELECT
        user_id,
        parent_asin,
        rating,
        timestamp
    FROM {SCHEMA}.{TABLE_RAW_REVIEWS}
    WHERE user_id = '{item}';
    """

    query_duration, _ = measure_clickhouse_query_performance(query)
    query_duration_logs.append(query_duration)
    
print(f"Over {len(random_product_list)} runs:")
print(f"Median query duration: {np.median(query_duration_logs)} ms")
print(f"Mean query duration: {np.mean(query_duration_logs)} ms")

Over 10 runs:
Median query duration: 13.0 ms
Mean query duration: 12.7 ms


## Query 5: Retrieve the average rating, rating distribution (count per star level), and total number of reviews for a given product.

In [10]:
# Record query duration
query_duration_logs = []
for item in random_product_list:
    query = f"""
    SELECT
        parent_asin,
        avg(rating)                          AS avg_rating,
        count()                              AS total_reviews,
        countIf(rating = 1)                  AS one_star_reviews,
        countIf(rating = 2)                  AS two_star_reviews,
        countIf(rating = 3)                  AS three_star_reviews,
        countIf(rating = 4)                  AS four_star_reviews,
        countIf(rating = 5)                  AS five_star_reviews
    FROM {SCHEMA}.{TABLE_RAW_REVIEWS}
    WHERE parent_asin = '{item}'
    GROUP BY parent_asin;
    """

    query_duration, _ = measure_clickhouse_query_performance(query)
    query_duration_logs.append(query_duration)
    
print(f"Over {len(random_product_list)} runs:")
print(f"Median query duration: {np.median(query_duration_logs)} ms")
print(f"Mean query duration: {np.mean(query_duration_logs)} ms")

Over 10 runs:
Median query duration: 5.5 ms
Mean query duration: 7.3 ms


# Load `5core` data: train, valid, test

In [6]:
queries = []

for table in TABLE_5CORE_TIMESTAMP_W_HIS_DICT.values():
    table_query = [
        f"DROP TABLE IF EXISTS {SCHEMA}.{table};", # Drop table
        f"""CREATE TABLE IF NOT EXISTS {SCHEMA}.{table}
        (
            user_id String,
            parent_asin String,
            rating UInt8,
            timestamp UInt64,
            history Array(String),
            product_category String
        )
        ENGINE = MergeTree()
        ORDER BY (user_id, parent_asin)
        PRIMARY KEY (user_id, parent_asin);""" # Create products table
    ]
    queries.extend(table_query)

client = connect_clickhouse()

for i, query in enumerate(queries):
    client.command(query)
    print(f"Query {i+1}: Successful")
    
client.close()

Query 1: Successful
Query 2: Successful
Query 3: Successful
Query 4: Successful
Query 5: Successful
Query 6: Successful


In [7]:
client = connect_clickhouse()

columns = [
    "user_id",
    "parent_asin",
    "rating",
    "timestamp",
    "history",
    "product_category"
]

batch = []

for category in categories:
    for split, table in TABLE_5CORE_TIMESTAMP_W_HIS_DICT.items():
        try:
            data = load_hf_data(
                root_split="5core",
                data_type="timestamp_w_his",
                split=split,
                category=category,
            )
            for record in tqdm(data, desc=f"Loading {split} data from {category} category..."):
                batch.append([
                    record["user_id"],
                    record["parent_asin"],
                    int(float(record["rating"])),
                    record["timestamp"],
                    record['history'].split() if record["history"] else [],
                    category
                ])

                if len(batch) >= BATCH_SIZE:
                    client.insert(
                        f"{SCHEMA}.{table}",
                        batch,
                        column_names=columns
                    )
                    batch.clear()

            # Insert remaining rows
            if batch:
                client.insert(
                    f"{SCHEMA}.{table}",
                    batch,
                    column_names=columns
                )
                batch.clear()
        
        except Exception as e:
            print(e)
        
client.close()

Loading train data from All_Beauty category...: 2237it [00:00, 2971.13it/s]
Loading valid data from All_Beauty category...: 276it [00:00, 1240.41it/s]
Loading test data from All_Beauty category...: 22it [00:00, 54.41it/s]
Loading train data from Toys_and_Games category...: 3114947it [02:25, 21451.94it/s]
Loading valid data from Toys_and_Games category...: 306944it [00:19, 15773.49it/s]
Loading test data from Toys_and_Games category...: 439995it [00:30, 14467.98it/s]
Loading train data from Cell_Phones_and_Accessories category...: 2208880it [01:23, 26424.51it/s]
Loading valid data from Cell_Phones_and_Accessories category...: 239437it [00:10, 22125.04it/s]
Loading test data from Cell_Phones_and_Accessories category...: 304468it [00:16, 18985.99it/s]
Loading train data from Industrial_and_Scientific category...: 297922it [00:12, 23192.98it/s]
Loading valid data from Industrial_and_Scientific category...: 44128it [00:02, 15218.86it/s]
Loading test data from Industrial_and_Scientific categ

BuilderConfig '5core_timestamp_w_his_Handmade_Products' not found. Available: ['raw_meta_All_Beauty', 'raw_meta_Toys_and_Games', 'raw_meta_Cell_Phones_and_Accessories', 'raw_meta_Industrial_and_Scientific', 'raw_meta_Gift_Cards', 'raw_meta_Musical_Instruments', 'raw_meta_Electronics', 'raw_meta_Handmade_Products', 'raw_meta_Arts_Crafts_and_Sewing', 'raw_meta_Baby_Products', 'raw_meta_Health_and_Household', 'raw_meta_Office_Products', 'raw_meta_Digital_Music', 'raw_meta_Grocery_and_Gourmet_Food', 'raw_meta_Sports_and_Outdoors', 'raw_meta_Home_and_Kitchen', 'raw_meta_Subscription_Boxes', 'raw_meta_Tools_and_Home_Improvement', 'raw_meta_Pet_Supplies', 'raw_meta_Video_Games', 'raw_meta_Kindle_Store', 'raw_meta_Clothing_Shoes_and_Jewelry', 'raw_meta_Patio_Lawn_and_Garden', 'raw_meta_Unknown', 'raw_meta_Books', 'raw_meta_Automotive', 'raw_meta_CDs_and_Vinyl', 'raw_meta_Beauty_and_Personal_Care', 'raw_meta_Amazon_Fashion', 'raw_meta_Magazine_Subscriptions', 'raw_meta_Software', 'raw_meta_Heal

Loading train data from Arts_Crafts_and_Sewing category...: 1382868it [01:11, 19256.71it/s]
Loading valid data from Arts_Crafts_and_Sewing category...: 192759it [00:12, 15005.93it/s]
Loading test data from Arts_Crafts_and_Sewing category...: 210810it [00:13, 15104.06it/s]
Loading train data from Baby_Products category...: 1067381it [00:52, 20268.17it/s]
Loading valid data from Baby_Products category...: 74245it [00:05, 14717.33it/s]
Loading test data from Baby_Products category...: 99457it [00:06, 14717.23it/s]


# Create `5core_timestamp_w_his_train_user_item_interactions` table

In [7]:
queries = [
    f"DROP TABLE IF EXISTS {SCHEMA}.{TABLE_5CORE_TIMESTAMP_W_HIS_TRAIN_USER_ITEM_INTERACTIONS};", # Drop table
    f"""CREATE TABLE IF NOT EXISTS {SCHEMA}.{TABLE_5CORE_TIMESTAMP_W_HIS_TRAIN_USER_ITEM_INTERACTIONS}
    (
        user_id String,
        interacted_items Array(String)
    )
    ENGINE = MergeTree()
    ORDER BY user_id"""
]

client = connect_clickhouse()

for i, query in enumerate(queries):
    client.command(query)
    print(f"Query {i+1}: Successful")

# limit history size (1 users can review an item multiple times)
ingest_query = f"""
INSERT INTO {SCHEMA}.{TABLE_5CORE_TIMESTAMP_W_HIS_TRAIN_USER_ITEM_INTERACTIONS}
SELECT
    user_id,
    arraySlice(
        groupUniqArray(parent_asin),
        1,
        500
    ) AS interacted_items
FROM {SCHEMA}.{TABLE_5CORE_TIMESTAMP_W_HIS_DICT['train']}
GROUP BY user_id
"""
client.command(ingest_query)

client.close()

Query 1: Successful
Query 2: Successful


# Create `recent_item_review_summary` table

In [2]:
queries = [
    f"DROP TABLE IF EXISTS {SCHEMA}.{TABLE_RECENT_ITEM_REVIEW_SUMMARY};", # Drop table
    f"""CREATE TABLE IF NOT EXISTS {SCHEMA}.{TABLE_RECENT_ITEM_REVIEW_SUMMARY}
    (
        parent_asin String,
        total_reviews UInt32,
        avg_rating Float32,
        45_star_reviews UInt32,
        12_star_reviews UInt32,
        helpful_votes UInt32,
        high_low_star_diff Int32
    )
    ENGINE = MergeTree()
    ORDER BY parent_asin"""
]

client = connect_clickhouse()

for i, query in enumerate(queries):
    client.command(query)
    print(f"Query {i+1}: Successful")

# limit history size (1 users can review an item multiple times)
ingest_query = f"""
INSERT INTO {SCHEMA}.{TABLE_RECENT_ITEM_REVIEW_SUMMARY}
SELECT
    parent_asin,
    count()                                     AS total_reviews,
    avg(rating)                                 AS avg_rating,
    countIf(rating >= 4)                        AS 45_star_reviews,
    countIf(rating <= 2)                        AS 12_star_reviews,
    sum(helpful_vote)                           AS helpful_votes,
    countIf(rating >= 4) - countIf(rating <= 2) AS high_low_star_diff
FROM
    (SELECT
        parent_asin,
        rating,
        helpful_vote
    FROM {SCHEMA}.{TABLE_RAW_REVIEWS}
    WHERE timestamp <= {END_TIMESTAMP_5CORE_TRAIN} AND timestamp >= {START_TIMESTAMP_5CORE_TRAIN})
GROUP BY parent_asin
"""
client.command(ingest_query)

client.close()

Query 1: Successful
Query 2: Successful
